In [ ]:
import torch
torch.cuda.empty_cache()

In [ ]:
import torch
import torch.nn as nn


def conv3x3(in_planes, out_planes, stride=1, groups=1, dilation=1):
    return nn.Conv2d(
        in_planes, out_planes, kernel_size=3, stride=stride,
        padding=dilation, groups=groups, bias=False, dilation=dilation
    )


def conv1x1(in_planes, out_planes, stride=1):
    return nn.Conv2d(in_planes, out_planes, kernel_size=1, stride=stride, bias=False)


class Bottleneck(nn.Module):
    expansion = 4

    def __init__(self, inplanes, planes, stride=1, downsample=None, groups=1, base_width=64, dilation=1):
        super(Bottleneck, self).__init__()
        width = int(planes * (base_width / 64.0)) * groups

        self.conv1 = conv1x1(inplanes, width)
        self.bn1 = nn.BatchNorm2d(width)

        self.conv2 = conv3x3(width, width, stride=stride, groups=groups, dilation=dilation)
        self.bn2 = nn.BatchNorm2d(width)

        self.conv3 = conv1x1(width, planes * self.expansion)
        self.bn3 = nn.BatchNorm2d(planes * self.expansion)

        self.relu = nn.ReLU(inplace=True)
        self.downsample = downsample
        self.stride = stride

    def forward(self, x):
        identity = x

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)
        out = self.relu(out)

        out = self.conv3(out)
        out = self.bn3(out)

        if self.downsample is not None:
            identity = self.downsample(x)

        out += identity
        out = self.relu(out)

        return out


class ResNet(nn.Module):
    def __init__(self, block, layers, num_classes=2, zero_init_residual=False, groups=1, width_per_group=64):
        super(ResNet, self).__init__()

        self.inplanes = 64
        self.dilation = 1
        self.groups = groups
        self.base_width = width_per_group

        self.conv1 = nn.Conv2d(3, self.inplanes, kernel_size=7, stride=2, padding=3, bias=False)
        self.bn1 = nn.BatchNorm2d(self.inplanes)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)

        self.layer1 = self._make_layer(block, 64, layers[0])
        self.layer2 = self._make_layer(block, 128, layers[1], stride=2)
        self.layer3 = self._make_layer(block, 256, layers[2], stride=2)
        self.layer4 = self._make_layer(block, 512, layers[3], stride=2)

        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(512 * block.expansion, num_classes)

        if zero_init_residual:
            for m in self.modules():
                if isinstance(m, Bottleneck):
                    nn.init.constant_(m.bn3.weight, 0)

    def _make_layer(self, block, planes, blocks, stride=1):
        downsample = None
        if stride != 1 or self.inplanes != planes * block.expansion:
            downsample = nn.Sequential(
                conv1x1(self.inplanes, planes * block.expansion, stride),
                nn.BatchNorm2d(planes * block.expansion),
            )

        layers = []
        layers.append(
            block(
                self.inplanes, planes, stride=stride, downsample=downsample,
                groups=self.groups, base_width=self.base_width, dilation=self.dilation
            )
        )
        self.inplanes = planes * block.expansion

        for _ in range(1, blocks):
            layers.append(
                block(
                    self.inplanes, planes,
                    groups=self.groups, base_width=self.base_width, dilation=self.dilation
                )
            )

        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)

        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)

        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.fc(x)

        return x


def resnet50():
    return ResNet(Bottleneck, [3, 4, 6, 3], num_classes=2)


In [ ]:
import torchvision.transforms as transforms

_IMAGENET_MEAN = [0.485, 0.456, 0.406]
_IMAGENET_STD = [0.229, 0.224, 0.225]

data_transforms = {
    'train': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomVerticalFlip(p=0.2),
        transforms.RandomRotation(degrees=15),
        transforms.ColorJitter(brightness=0.2, contrast=0.1, saturation=0.05, hue=0),
        transforms.RandomApply([transforms.GaussianBlur(kernel_size=3)], p=0.1),
        transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1)),
        transforms.ToTensor(),
        transforms.Normalize(mean=_IMAGENET_MEAN, std=_IMAGENET_STD)
    ]),
    'val': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=_IMAGENET_MEAN, std=_IMAGENET_STD)
    ])
}


In [ ]:
import os
import torch
from torch.utils.data import Dataset
import pandas as pd
from PIL import Image

class DeepfakeDataset(Dataset):
    def __init__(self, csv_file, root_dir, transform=None):
        self.annotations = pd.read_csv(csv_file).iloc[:, :2].copy()
        self.annotations.columns = ['image_id', 'label']
        self.annotations['image_id'] = self.annotations['image_id'].astype(int)
        self.annotations['label'] = self.annotations['label'].astype(int)
        self.root_dir = root_dir
        self.transform = transform

    def __len__(self):
        return len(self.annotations)

    def __getitem__(self, idx):
        if torch.is_tensor(idx):
            idx = idx.tolist()
        img_id = self.annotations.iloc[idx, 0]
        img_name = os.path.join(self.root_dir, f"{img_id}.jpg")
        image = Image.open(img_name).convert('RGB')
        label = self.annotations.iloc[idx, 1]
        label = torch.tensor(label, dtype=torch.long)
        if self.transform:
            image = self.transform(image)

        return image, label

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import StepLR
from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision.models import ResNet50_Weights

csv_path = '/kaggle/input/deepfakedata/dataset/train_solution.csv'
df = pd.read_csv(csv_path).iloc[:, :2].copy()
df.columns = ['image_id', 'label']
df['image_id'] = df['image_id'].astype(int)
df['label'] = df['label'].astype(int)

train_df, val_df = train_test_split(df, test_size=0.2, stratify=df['label'], random_state=42)
train_df.to_csv('train.csv', index=False)
val_df.to_csv('val.csv', index=False)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

use_pretrained = False
weights = None
model = resnet50()

if torch.cuda.device_count() > 1:
    print(f"{torch.cuda.device_count()}")
    model = nn.DataParallel(model)
else:
    print(f"{device}")

model = model.to(device)

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(model.parameters(), lr=0.0005, betas=(0.9, 0.999), eps=1e-08)

scheduler = StepLR(optimizer, step_size=26, gamma=0.1)

train_dataset = DeepfakeDataset(csv_file='train.csv',
                                root_dir='/kaggle/input/deepfakedata/dataset/train_images',
                                transform=data_transforms['train'])
val_dataset = DeepfakeDataset(csv_file='val.csv',
                              root_dir='/kaggle/input/deepfakedata/dataset/train_images',
                              transform=data_transforms['val'])

y = train_df['label'].values.astype(int)
class_counts = np.bincount(y)
class_weights = 1.0 / np.maximum(class_counts, 1)
sample_weights = class_weights[y]
sampler = WeightedRandomSampler(weights=torch.from_numpy(sample_weights).double(),
                                num_samples=len(sample_weights),
                                replacement=True)

train_loader = DataLoader(train_dataset, batch_size=64, sampler=sampler, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False, num_workers=2, pin_memory=True)

train_labels = train_dataset.annotations['label'].values.astype(int)
class_count = [int((train_labels == 0).sum()), int((train_labels == 1).sum())]
n_samples = sum(class_count)
w = [n_samples / (2 * max(1, x)) for x in class_count]
class_weights_t = torch.FloatTensor(w).to(device)
print(f"Class weights for WCE: {class_weights_t}")


In [ ]:
import time
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

from tqdm import tqdm

def train_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    all_preds = []
    all_labels = []

    progress_bar = tqdm(dataloader, desc="Training", leave=False)

    for images, labels in progress_bar:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

        progress_bar.set_postfix({'loss': f'{loss.item():.4f}'})

    epoch_loss = running_loss / len(dataloader.dataset)
    epoch_acc = accuracy_score(all_labels, all_preds)
    precision, recall, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average='binary')

    return epoch_loss, epoch_acc, f1, precision, recall

def validate(model, dataloader, criterion, device):
    model.eval()
    running_loss = 0.0

    all_probs = []
    all_labels = []

    with torch.no_grad():
        progress_bar = tqdm(dataloader, desc="Validating", leave=False)

        for images, labels in progress_bar:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)

            probs = torch.softmax(outputs, dim=1)[:, 1]
            all_probs.extend(probs.detach().cpu().numpy().tolist())
            all_labels.extend(labels.detach().cpu().numpy().tolist())

            progress_bar.set_postfix({'val_loss': f'{loss.item():.4f}'})

    epoch_loss = running_loss / len(dataloader.dataset)

    probs_np = np.asarray(all_probs, dtype=np.float32)
    labels_np = np.asarray(all_labels, dtype=np.int64)

    thresholds = np.linspace(0.01, 0.99, 99)
    best_f1, best_thr = -1.0, 0.5
    best_preds = None

    for thr in thresholds:
        preds = (probs_np >= thr).astype(np.int64)
        _, _, f1, _ = precision_recall_fscore_support(labels_np, preds, average='binary', zero_division=0)
        if f1 > best_f1:
            best_f1 = float(f1)
            best_thr = float(thr)
            best_preds = preds

    epoch_acc = accuracy_score(labels_np, best_preds)
    precision, recall, f1, _ = precision_recall_fscore_support(labels_np, best_preds, average='binary', zero_division=0)
    cm = confusion_matrix(labels_np, best_preds)

    return epoch_loss, epoch_acc, f1, precision, recall, cm, best_thr


In [ ]:
import torch
import matplotlib.pyplot as plt
import os
from tqdm import trange
from collections import OrderedDict
load_path = 'models/best_resnet50_model.pth'
resume_training = False

if resume_training and os.path.exists(load_path):
    print(f"Find model: {load_path}.")
    try:
        checkpoint = torch.load(load_path, map_location=device)
        state_dict = checkpoint['model_state_dict'] if isinstance(checkpoint, dict) else checkpoint
        new_state_dict = OrderedDict()
        is_model_parallel = isinstance(model, nn.DataParallel)
        for k, v in state_dict.items():
            if k.startswith('module.') and not is_model_parallel:
                name = k[7:]
            elif not k.startswith('module.') and is_model_parallel:
                name = 'module.' + k
            else:
                name = k
            new_state_dict[name] = v
            
        model.load_state_dict(new_state_dict)
        print("Load weigths")

    except Exception as e:
        print("From 0")

os.makedirs('models', exist_ok=True)
os.makedirs('plots', exist_ok=True)

num_epochs = 80
best_val_f1 = -1.0
best_threshold = 0.0

train_f1_history = []
val_f1_history = []
train_loss_history = []
val_loss_history = []

for epoch in range(num_epochs):
    print(f'Epoch {epoch+1}/{num_epochs}')
    print('-' * 10)
    
    train_loss, train_acc, train_f1, train_precision, train_recall = train_epoch(
        model, train_loader, criterion, optimizer, device)

    val_loss, val_acc, val_f1, val_precision, val_recall, cm, val_thr = validate(
        model, val_loader, criterion, device)

    print(f'Best threshold on val this epoch: {val_thr:.2f}')

    train_f1_history.append(train_f1)
    val_f1_history.append(val_f1)
    train_loss_history.append(train_loss)
    val_loss_history.append(val_loss)

    print(f'Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} F1: {train_f1:.4f}')
    print(f'Val Loss: {val_loss:.4f} Acc: {val_acc:.4f} F1: {val_f1:.4f}')
    print(f'Confusion Matrix:\n{cm}')

    scheduler.step()

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_threshold = val_thr
        best_model_path = 'models/best_resnet50_model.pth'
        model_to_save = model.module if hasattr(model, 'module') else model
        torch.save({'model_state_dict': model_to_save.state_dict(), 'threshold': val_thr}, best_model_path)
        print(f'New best model saved with Val F1: {val_f1:.4f} -> {best_model_path}')

    torch.save({'model_state_dict': model_to_save.state_dict(), 'threshold': val_thr}, 'models/last_resnet50_model.pth')
    plt.figure(figsize=(12, 5))
    
    plt.subplot(1, 2, 1)
    plt.plot(train_f1_history, label='Train F1', marker='o')
    plt.plot(val_f1_history, label='Val F1', marker='o')
    plt.title('F1-Score History')
    plt.xlabel('Epoch')
    plt.ylabel('F1-Score')
    plt.legend()
    plt.grid(True)

    plt.subplot(1, 2, 2)
    plt.plot(train_loss_history, label='Train Loss', marker='o')
    plt.plot(val_loss_history, label='Val Loss', marker='o')
    plt.title('Loss History')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True)

    plt.tight_layout()
    plot_path = f'plots/training_progress_epoch_{epoch+1}.png'
    plt.savefig(plot_path)
    plt.close() 
    print(f'Plots saved to {plot_path}')

print('Training complete.')
print(f'Best validation F1: {best_val_f1:.4f}')
print(f'Best threshold: {best_threshold:.2f}')


In [ ]:
import os
import torch
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import torchvision.transforms as T
import torchvision.transforms as transforms
from torchvision.models import ResNet50_Weights

class SubmissionDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.filenames = os.listdir(root_dir)
        self.transform = transform

    def __len__(self):
        return len(self.filenames)

    def __getitem__(self, idx):
        img_path = os.path.join(self.root_dir, self.filenames[idx])
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

use_pretrained = False
weights = ResNet50_Weights.DEFAULT if use_pretrained else None

val_transform = T.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

test_dataset = SubmissionDataset('/kaggle/input/deepfakedata/dataset/test_images', transform=val_transform)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False, num_workers=2)

model = resnet50().to(device)
checkpoint = torch.load('models/best_resnet50_model.pth', map_location=device)

state_dict = {k.replace('module.', ''): v for k, v in checkpoint['model_state_dict'].items()}
model.load_state_dict(state_dict)

threshold = checkpoint.get('threshold', 0.5)

model.eval()
predictions = []

with torch.no_grad():
    for images in tqdm(test_loader, desc="Test"):
        images = images.to(device)
        outputs = model(images)

        probs = torch.softmax(outputs, dim=1)[:, 1]
        preds = (probs >= threshold).long().cpu().numpy()

        predictions.extend(preds)


test_ids = [int(f.split('.')[0]) for f in test_dataset.filenames]

submission = pd.DataFrame({
    'id': test_ids,
    'target_feature': predictions
})

submission = submission.sort_values(by='id')
submission.to_csv('submission.csv', index=False)
print("Submission saved successfully!")
